# <h1 style="text-align: center;"> Presentation figure plotting notebook </h1>

This Python notebook loads MHWs netCDF files and produces figures.

## Setup

Run this script only once to setup all the necessary modules to generate the figures.

In [ ]:
# Enables modules autoreload (important during development)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# Advanced imports
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pymannkendall as mk

# Local imports
import balearic_mhws.config as config
import balearic_mhws.plotting.utils as utils

In [ ]:
from balearic_mhws.data.io import load_mhws, open_bathy

# Load the two MHW annual metrics dataset that were computed with REP and MEDREA respectively
ds_mhws_rep = load_mhws("yearly", "rep", False, "balears", (1987,2021))
ds_mhws_medrea = load_mhws("yearly", "medrea", False, "balears", (1987,2021))

# Load the MEDREA bathymetry dataset
ds_bathy_medrea = open_bathy()

# Interpolate bathymetry data onto REP grid
ds_bathy_rep = ds_bathy_medrea.interp(
    lat = ds_mhws_rep.lat,
    lon = ds_mhws_rep.lon,
    method = "linear"
)

In [ ]:
from balearic_mhws.plotting.utils import apply_regional_mask

# Compute regional masks
da_regions = xr.full_like(ds_mhws_medrea.isel(depth=0, year=0).total_days, np.nan)

continental_coast_mask = apply_regional_mask(da_regions, 'continental_coast', ds_bathy_medrea, return_mask=True)
balearic_coast_mask = apply_regional_mask(da_regions, 'balearic_coast', ds_bathy_medrea, return_mask=True)
balearic_sea_deep_mask = apply_regional_mask(da_regions, 'balearic_sea_deep', ds_bathy_medrea, return_mask=True)
west_algerian_deep_mask = apply_regional_mask(da_regions, 'west_algerian_deep', ds_bathy_medrea, return_mask=True)

# Compute a regional dataset with a number for each subregion
da_regions = da_regions.where(~continental_coast_mask, other=1)
da_regions = da_regions.where(~balearic_coast_mask, other=2)
da_regions = da_regions.where(~balearic_sea_deep_mask, other=3)
da_regions = da_regions.where(~west_algerian_deep_mask, other=4)

In [ ]:
from roman import toRoman

# Basic options
clim_period = (1987, 2021)
subplot_labels = [chr(ord('a')+n) + ')' for n in range(26)]
subplot_labels_roman = [toRoman(n) for n in range(101)]

## Splash slide

In [ ]:
from balearic_mhws.plotting.plot import plot_map

# Select total days metric derived from REP in 2022
da = ds_mhws_rep.total_days.sel(year=2022)

# Plotting the annual metric as a map
plot_map(
    # Parameter of data
    da.lon, da.lat,
    da,
    
    # Parameter of figure
    figsize = (12,4),
    fontsize = 12,
    figdpi = 200,

    # Parameters of graph
    yticks = 4,
    bottom_labels = False,
    left_labels = False,

    # Parameter of colorbar
    add_cbar = False,
    cmap = 'cmo.tempo',
    vlim = (20, 250),

    show_plots = True,
)

## Introduction

In [ ]:
from balearic_mhws.plotting.plot import plot_map

# Plot bathymetry map
plot_map(
    # Parameter of data
    ds_bathy_medrea.lon, ds_bathy_medrea.lat,
    ds_bathy_medrea.depth*0.001,

    # Parameter of figure
    figsize=(12,5),
    fontsize=12,
    figdpi=200,

    # Parameter of colorbar
    cbar_unit='Bathymetry [km]',
    cbar_orientation='horizontal',
    cbar_shrink=0.2,
    cbar_inversed=True,
    cbar_pad=0.06,
    cmap='cmo.deep',
    vlim=(0,3),

    # Parameters of graph
    extent=[-5.7,16.4,34.6,45],

    yticks=4,
    xticks=5,

    show_plots=True,
)

## Data

In [ ]:
from balearic_mhws.data.io import open_rep, open_medrea
from balearic_mhws.plotting.plot import plot_map, subplot

# Load a sample date of the raw REP and MEDREA datasets
da_rep = open_rep(time_selector='2010-01-01').T
da_medrea = open_medrea(time_selector='2010-01-01', depth_selector=None).T.isel(depth=0)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over models
for i, model in enumerate(['REP', 'MEDREA']):
    da = da_rep if model == 'REP' else da_medrea

    # Add the settings for the specific subplot 
    subplots_settings.append(dict(
        # Parameter of subplot
        pos = i+1,
        func = plot_map,

        # Parameter of data
        lon = da.lon, lat = da.lat,
        data = da,
        
        # Parameter of colorbar
        cbar_shrink = 0.3,
        cbar_ticks = 3,
        cbar_unit = 'T [°C]',
        cmap = 'cmo.thermal',
        vlim = (14, 17),

        # Parameter of graph
        xticks = 3,
        yticks = 4,

        left_labels = i==0,
    ))

subplot(
    # Parameter of subplot
    1,2,
    subplots_settings,

    # Parameter of figure
    figsize=(10,10),

    # Parameter of colorbar
    fig_cbar=True,
    fig_cbar_fraction=0.008,
    fig_cbar_ticks=3,
    fig_cbar_unit='[°C]',
    fig_cmap='cmo.thermal',
    fig_vmin=14, fig_vmax=17,

    # Must specify if the plot is a map
    fig_is_a_map = True,

    show_plots = True
)

In [ ]:
from balearic_mhws.data.io import open_medrea
from balearic_mhws.plotting.plot import plot_map

# Load a sample date of the raw MEDREA dataset
da_medrea = open_medrea(time_selector='2010-01-01', depth_selector=None).T

# Iterate over depths
for depth in [1, 50, 100, 500, 1000, 2500, 2800]:
    da = da_medrea.sel(depth=depth, method='nearest')

    plot_map(
        # Parameter of data
        da.lon, da.lat, da,

        # Parameter of figure
        figsize = (6,6),
        figdpi = 300,

        # Parameter of colorbar
        add_cbar = False,
        cmap = 'cmo.thermal',
        vlim = (12.860721, 16.5),

        # Parameter of graph
        xticks = 6,
        yticks = 4,
        bottom_labels = False,
        left_labels = False,
        
        # Parameter of saving
        save_path = f'medrea_{depth}m.png',
        save_plot = True
    )

## Methods - MHW detection

In [ ]:
from balearic_mhws.data.io import load_mhws

# Loads the region-averaged MHW dataset derived from REP
ds_mhws_rep_mean = load_mhws(
    ds_type = 'all_events',
    dataset_used = 'rep_mean',
    detrended = False,
    region = 'balears',
    clim_period = clim_period
)

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import scipy.ndimage as ndimage

from balearic_mhws.processing.compute_mhws import get_mhw_ts_from_ds
from balearic_mhws.plotting.plot import get_locator_from_ticks


# The ticks of the y axis
yticks = (5,0)
yticks_minor = (2.5,)

# The maximum category to be shown
max_thr = 2

# Iterate over showing categories
for show_categories in [False, True]:
    
    # Set the fontsize figure-wise
    with plt.rc_context({'font.size': 16}):
        # Convert raw dataset to a more friendly format
        time, mhws = get_mhw_ts_from_ds(ds_mhws_rep_mean, None, None, None)
        
        # Create matplotlib fig and ax
        fig = plt.figure(figsize=(8.4, 6), dpi=300, constrained_layout=True)
        ax = fig.add_subplot(1, 1, 1)

        # Plot the three main lines
        ax.plot(time, mhws["clim_sst"], label="REP-SST", c='k', lw=1)
        ax.plot(time, mhws["clim_thresh"], label="1987-2021 p90", c='r', ls='--', lw=1)
        ax.plot(time, mhws["clim_seas"], label="1987-2021 mean", c='r', lw=1)

        # Showing categories
        if show_categories:
            # Style for the 4 first categories
            colors = { 1: '#ffd86e', 2: '#ff621f', 3: '#df391b', 4: '#861a15'}
            lss = { 1: '--', 2: '-.', 3: ':', 4: '..'}

            # Compute thresholds for the categories
            for i in range(2, max_thr+2):
                mhws[f"clim_{i}thresh"] = mhws["clim_thresh"]*i - mhws["clim_seas"]*(i-1)
                ax.plot(time, mhws[f"clim_{i}thresh"], c="k", alpha=1, ls=lss[i], lw=1) 


        # Plot patches of color for each event and categories
        for ev in range(mhws["n_events"]):
            id_slice = slice(mhws["index_start"][ev], mhws["index_end"][ev]+1)
            temp = mhws["clim_sst"][id_slice]
            
            # Patches of color for category 1
            ax.fill_between(
                time[id_slice],
                mhws["clim_thresh"][id_slice],
                np.minimum(
                    mhws["clim_sst"][id_slice],
                    mhws["clim_2thresh"][id_slice],
                ) if show_categories else mhws["clim_sst"][id_slice],
                color=colors[1] if show_categories else '#faa307',
            )

            # Show yellow area under MHW events
            if not show_categories:
                ax.fill_between(
                    time[id_slice],
                    0,
                    mhws["clim_sst"][id_slice],
                    color="#ffd86e",
                    alpha=0.5,
                )

            # Showing categories
            if show_categories:
                # Patches of color for category > 1
                for i in range(2, max_thr+1):
                    lower_thr = mhws[f"clim_{i}thresh"][id_slice]
                    upper_thr = mhws[f"clim_{i+1}thresh"][id_slice]
                    times = time[id_slice]

                    exceed_bool = temp - lower_thr
                    exceed_bool = exceed_bool>=0

                    events, n_events = ndimage.label(exceed_bool)
                    
                    for ev_ in range(1,n_events+1):
                        slice_ = slice(np.where(events == ev_)[0][0]-1, np.where(events == ev_)[0][-1]+2)

                        if times[np.where(events == ev_)[0][0]].astype('datetime64[Y]').astype(int) + 1970 == 2022 and times[np.where(events == ev_)[0][0]].astype('datetime64[M]').astype(int) % 12 + 1 == 12:
                            print(f"Filling Cat {i} from {times[np.where(events == ev_)[0][0]]} to {times[np.where(events == ev_)[0][-1]]}")

                        ax.fill_between(
                            times[slice_],
                            lower_thr[slice_],
                            np.minimum(
                                temp[slice_],
                                upper_thr[slice_],
                            ),
                            color=colors[i],
                        )

        # matplotlib options
        ax.grid(ls='--', alpha=0.5)
        ax.grid(which='minor', ls='--', alpha=0.3)
        ax.set_ylabel("[°C]")

        if show_categories:
            ax.set_xlim(date(2022, 6, 1), date(2022, 7, 31))
            ax.set_ylim(22,31)
        
            ax.set_xticks(
                [date(2022, mm, 1) for mm in range(6,8)],
                ['Jun', 'Jul'],
            )
        
        else:
            ax.set_xlim(date(2022, 1, 1), date(2022, 12, 31))
            ax.set_ylim(12,31)
        
            ax.set_xticks(
                [date(2022, mm, 1) for mm in range(1, 13, 2)],
                ['Jan', 'Mar', 'May', 'Jul', 'Sep', 'Nov'],
            )
        
        ax.yaxis.set_major_locator(get_locator_from_ticks(yticks))
        ax.get_legend_handles_labels()

        if show_categories:
            ax.legend(
                [
                    Patch(color='#ff621f'),
                    Patch(color='#ffd86e'),
                    Line2D([0], [0], c='r', ls='--'),
                    Line2D([0], [0], c='r'),
                    Line2D([0], [0], color='k'),
                ], ["MHW category II", "MHW category I", "90$\mathregular{^{th}}$ percentile clim.", "Mean clim.", "Observed SST"],
                loc='upper left',
            )
        else:
            ax.legend(
                [
                    Patch(color='#faa307'),
                    Line2D([0], [0], c='r', ls='--'),
                    Line2D([0], [0], c='r'),
                    Line2D([0], [0], color='k'),
                ], ["Marine heatwaves", "90$\mathregular{^{th}}$ percentile clim.", "Mean climatology", "Daily REP-SST"],
                loc='upper left',
            )

        plt.show()


In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
year = 2022
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
]

# Size of the subplot grid
ncols = 1
nrows = len(stats)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over stats
for i, stat in enumerate(stats):
    row = i
    col = 0

    ds = ds_mhws_rep.sel(year=year)

    # Add the settings for the specific subplot
    subplots_settings.append(dict(
        # Parameter of subplot
        pos = row * ncols + col + 1,
        func = plot_map,

        # Parameter of data
        lon = ds.lon,
        lat = ds.lat,
        data = ds[stat],

        # Parameter of text
        title = config.mhws_stats_shortname[stat],

        # Parameter of colorbar
        cbar_shrink = 0.6,
        cbar_ticks = 2,
        cbar_unit = '' if stat == 'severity_mean_byday' else f'[{config.mhws_stats_units[stat]}]',
        cmap = mhws_stats_cmaps[stat],

        # Parameter of graph
        zero_to_nan = True,

        left_labels = False,
        bottom_labels = False,
        xticks = 6,
        yticks = 4,
    ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    pad_subplots=(0,0.6),

    # Parameter of figure
    figsize = (6,14),
    fig_fontsize = 14,
    figdpi = 200,

    # Must specify if the plot is a map
    fig_is_a_map = True,

    show_plots = True,
)

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np

from balearic_mhws.processing.compute_mhws import get_mhw_ts_from_ds


# Set the fontsize figure-wise
with plt.rc_context({'font.size': 12}):
    # Convert raw dataset to a more friendly format
    time, mhws = get_mhw_ts_from_ds(ds_mhws_rep_mean, None, None, None)

    # Create matplotlib fig and ax
    fig = plt.figure(figsize=(3, 2), dpi=400, constrained_layout=True)
    ax = fig.add_subplot(1, 1, 1)

    # Plot the three main lines
    ax.plot(time, mhws["clim_sst"], label="REP-SST", c='k', lw=1)
    ax.plot(time, mhws["clim_thresh"], label="1987-2021 p90", c='r', ls='--', lw=1)
    ax.plot(time, mhws["clim_seas"], label="1987-2021 mean", c='r', lw=1)

    # Patches of color for category 1
    for ev in range(mhws["n_events"]):
        id_slice = slice(mhws["index_start"][ev], mhws["index_end"][ev]+1)
        temp = mhws["clim_sst"][id_slice]
        
        ax.fill_between(
            time[id_slice],
            mhws["clim_thresh"][id_slice],
            mhws["clim_sst"][id_slice],
            color= "#ffd86e",
        )

    # matplotlib options
    ax.grid(ls='--', alpha=0.5)
    ax.grid(which='minor', ls='--', alpha=0.3)

    ax.set_xlim(date(2022, 1, 1), date(2022, 12, 31))
    ax.set_ylim(12,31)

    ax.tick_params(which='both', bottom=False, labelbottom=False, left=False, labelleft=False)

    plt.show()


## Methods - Subregions & Levels

In [ ]:
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap

from balearic_mhws.plotting.plot import plot_map

# Generate color map
alpha = 0.9
cmap = ListedColormap([('C0', alpha), ('C1', alpha), ('C2', alpha), ('C3', alpha)])

# Plot Balearic Islands region map
plot_map(
    # Parameter of data
    da_regions.lon, da_regions.lat,
    da_regions,

    # Parameter of figure
    figsize = (6, 6),
    fontsize = 14,
    figdpi = 200,

    # Parameter of colorbar
    add_cbar = False,
    cmap = cmap,
    vlim = (1, 4),

    # Parameters of graph
    extent = None,
    
    yticks=4,
    xticks=4,

    legend={
        'handles': [
            Patch(color='C0'),
            Patch(color='C1'),
            Patch(color='C2'),
            Patch(color='C3'),
        ],
        'labels': [utils.bold(txt) for txt in ["SPC", "BIC", "BS", "NWA"]],
        'loc': 'lower right',
        'handlelength': 0.7,
        'handletextpad': 0.4,
        'fontsize': 18,
    },

    show_plots=True
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a horizontal line
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]
y = np.linspace(0, 1, len(depths))

with plt.rc_context({'font.size': 14}):
    plt.figure(figsize=(1, 8), dpi=200)
    ax = plt.gca()

    ax.axes.get_xaxis().set_visible(False)
    ax.yaxis.set_inverted(True)
    plt.yticks(ticks=y, labels=depths)

    yticks = ax.get_yticks()

    new_labels = []
    for i, depth in enumerate(depths):
        text = ax.text(-0.1, yticks[i], depth,
                    va='center', ha='right',
                    transform=ax.get_yaxis_transform())

        if depth in [1, 203, 702, 2001]:
            text.set_fontsize(18)
            text.set_weight('bold')
        else:
            text.set_fontsize(14)

        new_labels.append(text)

    ax.set_yticklabels([])

    ax.spines['bottom'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.show()


## Surface MHWs

### Regional

In [ ]:
from scipy.stats import theilslopes

# Iterate over stats
for i, stat in enumerate([
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    # 'severity_mean_byday',
]):
    print(f"{i+1}. {config.mhws_stats_shortname[stat]}")

    model = "REP"

    da = ds_mhws_rep[stat].sel(year=slice(1982,2023))

    # Spatial averaging over the region
    y = da.mean(dim=["lon", "lat"])

    # Period averaging over 1982-2023 or 1987-2022
    # Using xarray functions
    mean = y.mean(dim='year')
    std = y.std(dim='year')

    print(f"{model} mean : {mean:.2f} ± {std:.2f} {config.mhws_stats_units[stat]}")

    # Calculation of trend over 1982-2023 or 1987-2022
    # Using Hamed Rao modified Mann-Kendall test for assessing significance of trend
    # Using Theil Sen estimator for trend slope
    # Using pymannkendall package to perform trend calculation
    trend, h, p, z, tau, s, var_s, slope, intercept = mk.hamed_rao_modification_test(y, alpha=0.05)
    
    # To calculate confidence interval (CI), needs to use scipy because pymannkendall
    # package doesn't include it
    slope_, intercept_, lower, upper = theilslopes(y, alpha=0.05)

    # h variable assesses the significance of the trend
    if h:
        print(f"{model} trend : {slope*10:.2f} ± {(upper-lower)/2 * 10:.2f} {config.mhws_stats_units[stat]}/decade (p-value={p:.5f})")
    
    else:
        print(f"No significant trend for {model}.")

    
    print()

### Subregional

In [ ]:
# Apply the Mann-Kendall test on the whole REP dataset
h, p, slope = xr.apply_ufunc(
    utils.apply_mk_test,
    ds_mhws_rep[config.mhws_basic_stats],
    input_core_dims = [["year"]],
    output_core_dims = [[], [], []],
    vectorize = True,
    dask = "parallelized",
    output_dtypes = [bool, float, float],
)

# Final dataset with decadal trends
ds_trend = slope.where(h) * 10

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
columns = ['Mean', 'Trend',]

# Iterates over subsets of stats
for stats in [
    'total_days',
    'duration',
], [
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    # 'severity_mean_byday',
]:
    # Iterate over columns
    for column in columns:

        # Size of the subplot grid
        ncols = len(stats)
        nrows = 1


        # Start creating individual subplot setting
        subplots_settings = []

        # Iterate over stats
        for iss, stat in enumerate(stats):
            col = iss
            row = 0

            cmap = mhws_stats_cmaps[stat]

            if column == 'Mean':
                da = ds_mhws_rep[stat].mean(dim='year')
            
            elif column == 'Trend':
                da = ds_trend[stat]
                cmap = 'cmo.speed'

            # Add the settings for the specific subplot
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row*ncols + col + 1,
                func = plot_map,

                # Parameter of data
                lon = da.lon,
                lat = da.lat,
                data = da,

                # Parameter of text
                title = utils.bold(config.mhws_stats_shortname[stat]),
                fontsize_title = 1.2,

                # Parameter of colorbar
                cbar_shrink = 0.7,
                cbar_pad = 0.015,
                cbar_unit = f"[{config.mhws_stats_units[stat] + ('/decade' if column == 'Trend' else '')}]",
                cbar_ticks = 3,
                cmap = cmap,

                # Parameter of contours
                contours_levels = [2.5, 3.5],
                contours_data = [
                    da_regions.lon,
                    da_regions.lat,
                    da_regions
                ],
                contours_kwargs = dict(
                    colors='k', linewidths=1, zorder=-1
                ),
                
                # Parameter of graph
                xticks = 3,
                yticks = 4,

                left_labels = (col == 0),
                bottom_labels = (row == nrows - 1),
            ))

        # Generate the figure from subplot settings
        subplot(
            # Parameter of subplot
            nrows, ncols,
            subplots_settings,
            
            # Parameter of figure
            figsize = (14.4,3) if ncols == 3 else (10,3),
            fig_fontsize = 16,
            figdpi = 200,

            # Must specify if the plot is a map
            fig_is_a_map = True,

            show_plots = True,
        )

## Subsurface MHWs

### Regional

In [ ]:
from textwrap import wrap

from balearic_mhws.plotting.plot import plot_bars, subplot


# Parameters to plot in the figure
columns = [
    'Mean',
    'Trend'
]

# Options 
colors = {
    "total_days":           "#13847b",
    "duration":             "#e15f52",
    "total_icum":           "#ca8f18",
    "intensity_max_max":    "#af2f24",
    'intensity_mean_byday': "#af2f24",
    'severity_mean_byday':  "#c96716",
}


# Iterate over subsets of stats
for stats in [
    [
        'total_days',
        'duration',
    ], [
        'total_icum',
        'intensity_max_max',
        'intensity_mean_byday',
        # 'severity_mean_byday',
    ]
]:
    # Iterate over columns
    for column in columns:

        # Size of the subplot grid
        ncols = len(stats)
        nrows = 1


        # Start creating individual subplot setting
        subplots_settings = []

        # Iterate over stats
        for col, stat in enumerate(stats):
            row = 0

            ds = ds_mhws_medrea

            std, hatchs = None, None

            if column == 'Mean':
                da = ds[stat].mean(dim=['lon', 'lat']).mean(dim='year')
                std = ds[stat].mean(dim=['lon', 'lat']).std(dim='year')

            elif column == 'Trend':
                da = ds[stat].mean(dim=['lon', 'lat'])

                h, p, slope = xr.apply_ufunc(
                    utils.apply_mk_test,
                    da,
                    input_core_dims=[["year"]],
                    output_core_dims=[[], [], []],
                    vectorize=True,
                    dask="parallelized",
                    output_dtypes=[bool, float, float],
                )

                # Change from yearly trend to decadly trend
                da = slope.where(h, 0) * 10

            # Add the settings for the specific subplot 
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row*ncols + col + 1,
                func = plot_bars,
                
                # Parameter of data
                depths = da.depth.values,
                vars = {'a': da},
                colors = colors[stat],

                # Parameter of text
                title = utils.bold('\n'.join(wrap(config.mhws_stats_shortname[stat], 17))),
                xlabel = f"[{config.mhws_stats_units[stat] + ('/decade' if column == 'Trend' else '')}]",

                # Parameter of graph
                bars_pad = 0.5,

                xticks = 4,
                yticks = 2,

                left_labels = (col == 0),

            ))
        
        # Generate the figure from subplot settings
        subplot(
            # Parameter of subplot
            nrows, ncols,
            subplots_settings,
            
            # Parameter of figure
            figsize = (10, 4) if ncols == 3 else (7.4, 4),
            fig_fontsize = 16,
            figdpi = 200,

            show_plots = True,
        )

### Subregional

In [ ]:
# Generic one

from balearic_mhws.plotting.plot import plot_timeserie, subplot
from balearic_mhws.plotting.utils import apply_regional_mask

# Parameters to plot in the figure
subsets = [
    (['total_days', 'duration'], [702, 1005, 1487, 2001]),
    (['intensity_max_max'], [51, 98])
]
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]

# Create subregional datasets
regions_ds = {}

for region in regions:
    regions_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)

# Iterate over subsets of stats and depths
for stats, depths in subsets:
    # Size of the subplot grid
    ncols = len(stats)
    nrows = len(depths)


    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over stats and depths
    for iss, stat in enumerate(stats):
        for id, depth in enumerate(depths):
            col = iss
            row = id

            # Add the settings for the specific subplot
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row * ncols + col + 1,
                func = plot_timeserie,

                # Parameter of data
                vars = {
                    config.region_shortname[region]:
                        regions_ds[region][stat].sel(depth=depth, method='nearest').mean(dim=["lon","lat"], keep_attrs=True)
                    
                    for region in (
                        regions[2:]
                        if (isinstance(depth, float) and depth > 200)
                        else
                            regions
                    )
                },
                times = regions_ds[regions[0]].year,
                colors = {
                    "Continental coast": 'C0',
                    "Balearic Islands coast": 'C1',
                    "Balearic Sea deep": 'C2',
                    "West Algerian Basin deep": 'C3',
                },

                # Parameer of text
                title = utils.bold(config.mhws_stats_shortname[stat]) + 
                        ('' if stat == 'severity_mean_byday' else f" [{config.mhws_stats_units[stat]}]") if row == 0 else None,
                ylabel = fr"$\bf{{{depth:.0f}m}}$" if col == 0 else None,

                # Parameter of graph
                legend = False,
                nans_to_zero=True,
                ylim = (0, 366) if stat == 'total_days' else (0, 270) if stat == 'duration' else None,

                xticks = (10,),
                xticks_minor = 2,
                yticks = 3 if stat =='duration' else 3,

                bottom_labels = (row == nrows-1),
            ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,

        # Parameter of figure
        figsize = (8, 4.2) if len(stats) == 2 else (4.2, 2.6),
        figdpi = 200,
        fig_fontsize = 14,

        show_plots = True,
    )

In [ ]:
from textwrap import wrap
from matplotlib.ticker import StrMethodFormatter

from balearic_mhws.plotting.plot import plot_vertical_mean, subplot
from balearic_mhws.plotting.utils import apply_regional_mask


# Parameters to plot in the figure
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
]

# Create subregional datasets
regions_ds = {}

for region in regions:
    regions_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)

# Iterate over columns
for averaging in ['Mean', 'Trend']:
    # Size of the subplot grid
    ncols = len(stats)
    nrows = 1
    

    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over stats
    for iss, stat in enumerate(stats):
        col = iss
        row = 0

        region_das = {}

        for region in regions:
            ds = regions_ds[region]

            if averaging == 'Mean':
                da = ds[stat].mean(dim=['lon', 'lat']).mean(dim='year')
     
            elif averaging == 'Trend':
                da = ds[stat].mean(dim=['lon', 'lat'])

                h, p, slope = xr.apply_ufunc(
                    utils.apply_mk_test,
                    da,
                    input_core_dims=[["year"]],
                    output_core_dims=[[], [], []],
                    vectorize=True,
                    dask="parallelized",
                    output_dtypes=[bool, float, float],
                )

                # Change from yearly trend to decadly trend
                da = slope.where(h, 0) * 10
            
            region_das[region] = da

        # Add the settings for the specific subplot
        subplots_settings.append(dict(
            # Parameter of subplot
            pos = row * ncols + col + 1,
            func = plot_vertical_mean,
            
            # Parameter of data
            depths = depths,
            vars = region_das,
            
            # Parameter of text
            title = utils.bold('\n'.join(wrap(config.mhws_stats_shortname[stat], width=9 if stat == 'total_days' else 10))),
            unit = config.mhws_stats_units[stat],

            # Parameter of graph
            ylim = (1, 2001),
            xticks = 3,
            yticks = [1, 203, 702, 2001],
            yticks_minor = [51, 98, 153, 493, 1005,  1487,],
            yticks_formatter = StrMethodFormatter("{x:.0f}m"),

            bottom_labels = (row == nrows-1),
            left_labels = (col == 0),
        ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,

        # Parameter of figure
        figsize = (8, 4.2),
        figdpi = 200,
        fig_fontsize = 14,

        show_plots = True,
    )

## Drivers

In [ ]:
from balearic_mhws.plotting.plot import plot_timeserie, subplot


# Parameters to plot in the figure
depths = [1,  98, 203, 702, 1487,]
stat = 'total_days'

# Size of the subplot grid
ncols = 1
nrows = len(depths)


# Start creating individual subplot setting
subplots_settings = []

# Iterate over depths
for id, depth in enumerate(depths):
    col = id // nrows 
    row = id % nrows

    # Add the settings for the specific subplot 
    subplots_settings.append(dict(
        # Parameter of subplot
        pos = row * ncols + col + 1,
        func = plot_timeserie,

        # Parameter of data
        vars = ds_mhws_medrea[stat].sel(depth=depth, method='nearest').mean(dim=["lon","lat"], keep_attrs=True),
        times = ds_mhws_medrea.year,
        colors = {
            "total_days":           "#13847b",
            "duration":             "#e15f52",
            "total_icum":           "#ca8f18",
            "intensity_max_max":    "#af2f24",
            'intensity_mean_byday': "#af2f24",
            'severity_mean_byday':  "#c96716",
        }[stat],

        # Parameter of text
        ylabel = fr"$\bf{{{depth:.0f}m}}$" if col == 0 else None,

        # Parameter of graph
        nans_to_zero=True,
        legend=False,
        ylim = (0,365),

        xticks = (10,),
        xticks_minor = 2,
        yticks = 3,

        bottom_labels = (row == nrows-1),
    ))

# Generate the figure from subplot settings
subplot(
    # Parameter of subplot
    nrows, ncols,
    subplots_settings,
    pad_subplots = (0, 0.6),

    # Parameter of figure
    figsize = (5, 10),
    fig_title = utils.bold(config.mhws_stats_shortname[stat]) + f" [{config.mhws_stats_units[stat]}]",
    figdpi = 200,
    fig_fontsize = 14,

    show_plots = True,
)

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
subsets = [
    ([2003,2022], [1, 51]),
    ([1998, 2008, 2017], [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]),
    ([2004], [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]),
]
stat = 'total_days'

# Iterate over subsets of years and depths
for years, depths in subsets:
    # Size of the subplot grid
    ncols = len(years)
    nrows = len(depths)


    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over depths and years
    for id, depth in enumerate(depths):
        for iy, year in enumerate(years):
            row = id
            col = iy

            ds = ds_mhws_medrea.sel(year=year, depth=depth, method='nearest')

            # Add the settings for the specific subplot
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row * ncols + col + 1,
                func = plot_map,

                # Parameter of data
                lon = ds.lon,
                lat = ds.lat,
                data = ds[stat],

                # Parameter of text
                title = utils.bold(year) if row == 0 else None,
                ylabel = utils.bold(f"{depth:.0f}m") if col == 0 else None,
                ylabel_pad = -0.02,

                # Parameter of graph
                zero_to_nan = True,

                xticks = 6,
                yticks = 4,
                left_labels = False,
                bottom_labels = False,
            ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,
        
        # Parameter of figure
        figsize = (5, 3) if 2003 in years else (5, 11.2),
        fig_fontsize = 14,
        figdpi = 200,

        # Parameter of colobar
        fig_cbar = True,
        fig_cmap = mhws_stats_cmaps[stat],
        fig_cbar_unit = utils.bold(config.mhws_stats_shortname[stat]) + f" [{config.mhws_stats_units[stat]}]",
        fig_cbar_fraction = 0.03,
        fig_cbar_pad = 0.015,
        fig_cbar_ticks = 4,

        # Must specify if the plot is a map
        fig_is_a_map = True,

        show_plots = True,
    )

## Appendices

### Appendix - Global overview

In [ ]:
from balearic_mhws.plotting.plot import plot_map, subplot, mhws_stats_cmaps

# Parameters to plot in the figure
years = range(1987, 2023)
depths = [1, 51, 98, 153, 203, 493, 702, 1005, 1487, 2001]

# Iterate over stats
for stat in [
    'total_days',
]:
    # Size of the subplot grid
    ncols = len(years)
    nrows = len(depths)


    # Start creating individual subplot setting
    subplots_settings = []

    # Iterate over depths and years
    for id, depth in enumerate(depths):
        for iy, year in enumerate(years):
            row = id
            col = iy

            ds = ds_mhws_medrea.sel(year=year, depth=depth, method='nearest')

            # Add the settings for the specific subplot 
            subplots_settings.append(dict(
                # Parameter of subplot
                pos = row * ncols + col + 1,
                func = plot_map,

                # Parameter of data
                lon = ds.lon,
                lat = ds.lat,
                data = ds[stat],

                # Parameter of text
                title = year if row == 0 and (year % 2 == 0) else None,
                ylabel = f"{depth:.0f}m" if col == 0 and row % 2 == 0 else None,
                ylabel_pad=-0.08,

                # Parameter of graph
                zero_to_nan = True,

                left_labels = False,
                bottom_labels = False,
                xticks = 6,
                yticks = 4,
            ))

    # Generate the figure from subplot settings
    subplot(
        # Parameter of subplot
        nrows, ncols,
        subplots_settings,
        
        # Parameter of figure
        subplotsize = (6.6, 5),
        figdpi = 30,
        fig_fontsize = 120,
        pad_subplots = (0.05, 0.05),

        # Parameter of colorbar
        fig_cbar = True,
        fig_cbar_unit = f"[{config.mhws_stats_units[stat]}]",
        fig_cbar_fraction = 0.05,
        fig_cbar_pad = 0.01,
        fig_cbar_orientation='horizontal',
        fig_cbar_ticks = 5,
        fig_cmap = mhws_stats_cmaps[stat],

        # Must specify if the plot is a map
        fig_is_a_map = True,

        show_plots = True,
    )

### Appendix - Time series of temperature

In [ ]:
from balearic_mhws.data.io import load_mhws

# Loads the region-averaged MHW dataset derived from MEDREA
ds_mhws_medrea_mean = load_mhws(
    ds_type = 'all_events',
    dataset_used = 'medrea_mean',
    detrended = False,
    region = 'balears',
    clim_period = clim_period
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import date

from balearic_mhws.processing.compute_mhws import get_mhw_ts_from_ds
from balearic_mhws.plotting.plot import get_locator_from_ticks


# Parameters to plot in the figure
depths = [1, 98, 203, 702, 1487]
date_ranges = [(0.5, 39.31), (0.5, 39.31)]

# Set the fontsize figure-wise
with plt.rc_context({'font.size': 18}):
    # Create matplotlib fig
    fig = plt.figure(figsize=(18, 9), dpi=200, constrained_layout=True)

    # Size of the subplot grid
    nrows = len(depths)
    ncols = len(date_ranges)+1

    # Iterate over dates and depths
    for col, (t0, t1) in enumerate(date_ranges):
        for row, depth in enumerate(depths):
            # Convert raw dataset to a more friendly format
            time, mhws = get_mhw_ts_from_ds(ds_mhws_medrea_mean, None, None, depth)

            # If there is no MHW event, let's skip it
            if mhws == -1:
                continue

            index = (row*ncols + 1, row*ncols + 2) if col == 0 else row*ncols + col + 2

            # Create matplotlib ax
            ax = fig.add_subplot(nrows, ncols, index)

            # Plot the three main lines 
            ax.plot(time, mhws["clim_sst"], label="thetao", c='k', lw=1)
            ax.plot(time, mhws["clim_thresh"], label="1987-2021 p90", c='r', ls='--', lw=1)
            ax.plot(time, mhws["clim_seas"], label="1987-2021 mean", c='r', lw=1)

            # Compute thresholds for the categories
            mhws["clim_2thresh"] = mhws["clim_thresh"]*2 - mhws["clim_seas"]
            mhws["clim_3thresh"] = mhws["clim_thresh"]*3 - mhws["clim_seas"]*2

            # Plot patches of color for each event and categories
            for ev in range(mhws["n_events"]):
                id_slice = slice(mhws["index_start"][ev], mhws["index_end"][ev]+1)

                # Patches of color for category 1
                ax.fill_between(
                    time[id_slice],
                    mhws["clim_thresh"][id_slice],
                    np.minimum(
                        mhws["clim_sst"][id_slice],
                        mhws["clim_2thresh"][id_slice],
                    ),
                    color='#ffd86e',
                )
                
                # Patches of color for category 2
                strong_intensity = mhws["clim_sst"][id_slice]
                strong_intensity[np.where(mhws["clim_sst"][id_slice] < mhws["clim_2thresh"][id_slice])] = np.nan

                ax.fill_between(
                    time[id_slice],
                    mhws["clim_2thresh"][id_slice],
                    np.minimum(
                        strong_intensity,
                        mhws["clim_3thresh"][id_slice],
                    ),
                    color='#ff621f',
                )
                
                # Patches of color for category 3
                severe_intensity = mhws["clim_sst"][id_slice]
                severe_intensity[np.where(mhws["clim_sst"][id_slice] < mhws["clim_3thresh"][id_slice])] = np.nan

                ax.fill_between(
                    time[id_slice],
                    mhws["clim_3thresh"][id_slice],
                    severe_intensity,
                    color='#df391b',
                )

            # matplotlib options
            ax.grid(True, ls='-', alpha=0.8)
            ax.grid(True, ls='--', alpha=0.5, which='minor')
            ax.set_title(fr"$\bf{{{depth:.0f}m}}$")
            
            ax.yaxis.set_major_locator(get_locator_from_ticks(1))

            if col == 0:
                ax.set_ylabel('[°C]')
            else:
                ax.tick_params(which='both', left=False, labelleft=False)
            
            fig.align_ylabels()

            if col == 1:
                ax.set_xlim(date(2020, 1, 1), date(2022, 12, 31))
                ax.set_xticks(
                    [date(2021, 1, 1), date(2022, 1, 1)],
                    ["2021", "2022"]
                )
                ax.set_xticks(
                    [date(2020, 7, 1), date(2021, 7, 1), date(2022, 7, 1)],
                    minor=True,
                )

            else:
                ax.set_xlim(date(1987, 1, 1), date(2022, 12, 31))
                ax.set_xticks(
                    [date(1990, 1, 1), date(2000, 1, 1), date(2010, 1, 1), date(2020, 1, 1)],
                    ["1990", "2000", "2010", "2020"]
                )
                ax.set_xticks(
                    [date(1995, 1, 1), date(2005, 1, 1), date(2015, 1, 1)],
                    minor=True,
                )
                
            if not row == nrows-1:
                ax.tick_params(which='both', bottom=False, labelbottom=False)

            ax.get_legend_handles_labels()

plt.show()

### Appendix - Time series of annual metrics

In [ ]:
# Generic one

from balearic_mhws.plotting.plot import plot_timeserie, subplot
from balearic_mhws.plotting.utils import apply_regional_mask

regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]
depths = [0, 50, 100, 150, 200, 500, 700, 1000, 1500, 2000,]

names = {
    "total_days":           r"$\bf{Total  \ days}$ [days]",
    "duration":             r"$\bf{Mean \ duration}$ [days]",
    "total_icum":           r"$\bf{Cumulative \ intensity}$ [°C.day]",
    "intensity_max_max":    r"$\bf{Maximum \ intensity}$ [°C]",
    "intensity_mean_byday": r"$\bf{Mean \ intensity}$ [°C]",
    "severity_mean_byday": r"$\bf{Mean \ severity}$",
}

regions_ds = {}

for region in regions:
    regions_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)

for i, depth in enumerate(depths):
    depths[i] = ds_mhws_medrea.sel(depth=depth, method='nearest').depth.item()


for stats in [[
    'total_days',
    'duration',
    'total_icum',
# ],[
    'intensity_max_max',
    'intensity_mean_byday',
    # 'severity_mean_byday',
]]:
    ncols = len(stats)
    nrows = len(depths)
    
    subplots_settings = []

    for iss, stat in enumerate(stats):
        for id, depth in enumerate(depths):
            col = iss
            row = id
            
            if stat in ['intensity_max_max', 'intensity_mean_byday', 'severity_mean_byday']:
                subplot_label_ha = 'right'

                if depth > 190 and depth < 210:
                    subplot_label_va = 'top'
                else:
                    subplot_label_va = 'bottom'
            
            else:
                subplot_label_ha = 'left'
                subplot_label_va = 'top'

            subplots_settings.append(dict(
                pos = row * ncols + col + 1,
                func = plot_timeserie,
                vars = {
                    config.region_shortname[region]:
                        regions_ds[region][stat].sel(depth=depth).mean(dim=["lon","lat"], keep_attrs=True)
                    
                    for region in (
                        regions[2:]
                        if (isinstance(depth, float) and depth > 200)
                        else
                            regions
                    )
                },
                times = regions_ds[regions[0]].year,
                colors = {
                    "Continental coast": 'C0',
                    "Balearic Islands coast": 'C1',
                    "Balearic Sea deep": 'C2',
                    "West Algerian Basin deep": 'C3',
                },

                nans_to_zero=True,

                title = names[stat] if row == 0 else None,
                ylabel = fr"$\bf{{{depth:.0f} \ m}}$" if col == 0 else None,
                legend=False,

                xticks=(10,),
                xticks_minor=2,
                yticks=3,

                bottom_labels = (row == nrows-1),

                texts = [dict(
                    x=(0.02-0.98) * (subplot_label_ha == 'left') + 0.98,
                    y=(0.04-0.96) * (subplot_label_va == 'bottom') + 0.96,
                    s=utils.bold(subplot_labels_roman[row + nrows*col + 1].lower() + ')'),
                    ha=subplot_label_ha,
                    va=subplot_label_va,
                    fontsize=18
                )]
            ))

    fig = subplot(
        nrows, ncols,
        subplots_settings,

        figsize = (24, 11),
        figdpi = 200,
        fig_fontsize = 14,

        show_plots = True,
    )

### Appendix - Subsurf subreg bar plots

In [ ]:
from balearic_mhws.plotting.plot import plot_bars, subplot
from balearic_mhws.plotting.utils import apply_regional_mask

# USER INPUT - Choose what to be displayed
columns = [
    'Mean',
    # '2022',
    'Trend'
]
stats = [
    'total_days',
    'duration',
    'total_icum',
    'intensity_max_max',
    'intensity_mean_byday',
    'severity_mean_byday',
]
regions = [
    "continental_coast",
    "balearic_coast",
    "balearic_sea_deep",
    "west_algerian_deep",
]

# Names of stat with bolding
names = {
    "total_days":           r"$\bf{Total}$" "\n" r"$\bf{days}$ [days]",
    "duration":             r"$\bf{Mean}$" "\n" r"$\bf{duration}$ [days]",
    "total_icum":           r"$\bf{Cumulative}$" "\n" r"$\bf{intensity}$" "\n[°C.days]",
    "intensity_max_max":    r"$\bf{Maximum}$" "\n" r"$\bf{intensity}$ [°C]",
    "intensity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{intensity}$ [°C]",
    "severity_mean_byday": r"$\bf{Mean}$" "\n" r"$\bf{severity}$",
}

subplot_label_pos = {
    "total_days":           [1, 1],
    "duration":             [1, 1],
    "total_icum":           [0, 0],
    "intensity_max_max":    [0, 0],
    "intensity_mean_byday": [0, 0],
    "severity_mean_byday":  [0, 0],
}


regions_ds = {}

for region in regions:
    regions_ds[region] = apply_regional_mask(ds_mhws_medrea, region, ds_bathy_medrea)

subplots_settings = []

ncols = len(columns)
nrows = len(stats)

for row, stat in enumerate(stats):
    for col, column in enumerate(columns):
        region_das = {}

        for region in regions:
            ds = regions_ds[region].sel(depth=[0, 100, 200, 700, 1500], method='nearest')

            if column == 'Mean':
                da = ds[stat].mean(dim=['lon', 'lat']).mean(dim='year')
            
            elif column == '2022':
                da = ds[stat].sel(year=2022).mean(dim=['lon', 'lat'])
     
            elif column == 'Trend':
                da = ds[stat].mean(dim=['lon', 'lat'])

                def apply_mk_test(y):
                    if np.isnan(y).all():
                        return np.nan, np.nan, np.nan
                    
                    trend, h, p, z, tau, s, var_s, slope, intercept = mk.hamed_rao_modification_test(y, alpha=0.05)
                    
                    return h, p, slope

                h, p, slope = xr.apply_ufunc(
                    apply_mk_test,
                    da,
                    input_core_dims=[["year"]],
                    output_core_dims=[[], [], []],
                    vectorize=True,
                    dask="parallelized",
                    output_dtypes=[bool, float, float],
                )

                da = slope.where(h, 0)

                # Change from yearly trend to decadly trend
                da = da * 10

                unit = f"[{config.mhws_stats_units[stat]} /decade]"
            
            region_das[region] = da


        subplots_settings.append(dict(
            pos = row*ncols + col + 1,
            func = plot_bars,
            depths = da.depth.values,
            vars = region_das,

            title = utils.bold(column) if row == 0 else None,
            ylabel = names[stat] if col == 0 else None,

            # hatch = region_hatchs if column == 'Trend' else None,

            bars_pad = 1.2,
            
            xticks = 5,

            left_labels = (col == 0),

            texts = [dict(
                x=0.98,
                y=(0.96-0.04) * subplot_label_pos[stat][col] + 0.04,
                s=utils.bold(subplot_labels[row + nrows*col]),
                ha='right',
                va='top' if subplot_label_pos[stat][col] else 'bottom',
                fontsize=18
            )]
        ))

# Creates figure settings, using user settings firstly, otherwise use default
subplot(
    nrows, ncols,
    subplots_settings,
    
    figsize=(12,12),
    fig_fontsize = 14,
    figdpi=200,

    show_plots = True,
)